# FastText

这是FastText模型的从零开始实现，简化了底层的分层softmax（在连续词袋模型实现过）。

In [1]:
# In[1]: 导入库与定义 FastText 官方哈希算法
import torch
import torch.nn as nn
import torch.optim as optim

def fnv1a_hash(string):
    """
    复刻 FastText C++ 底层使用的 FNV-1a 哈希算法。
    确保无论在什么环境下，相同的字符串都会得到相同的 uint32 哈希值。
    """
    h = 2166136261
    for char in string:
        # 将字符转为字节并异或
        h = h ^ ord(char)
        # 乘以 FNV 质数并截断为 32 位无符号整数
        h = (h * 16777619) & 0xffffffff
    return h

In [2]:
# In[2]: 定义特征提取器

class FastTextFeatureExtractor:
    def __init__(self, vocab, bucket_size=2000000, minn=3, maxn=6, word_ngrams=2):
        self.vocab = vocab              # 精确词表 mapping: {word: id}
        self.vocab_size = len(vocab)
        self.bucket_size = bucket_size  # 哈希桶大小 (官方默认 200万)
        self.minn = minn                # 字符 n-gram 最小长度
        self.maxn = maxn                # 字符 n-gram 最大长度
        self.word_ngrams = word_ngrams  # 词级别 n-gram 长度
        
    def get_features(self, text):
        """
        将一段文本转换为 ID 列表，包含：
        1. 词表 ID
        2. 字符 n-gram 的哈希 ID (映射到 Bucket)
        3. 词级别 n-gram 的哈希 ID (映射到 Bucket)
        """
        words = text.lower().split()
        feature_ids = []
        
        # 1. 获取完整单词的 ID 和 字符 n-gram ID
        for word in words:
            # 完整单词：如果存在于词表，取其精确 ID；否则忽略（后续靠其字符 n-gram 兜底）
            if word in self.vocab:
                feature_ids.append(self.vocab[word])
            
            # 字符 n-gram：添加边界符
            w_padded = f"<{word}>"
            for n in range(self.minn, self.maxn + 1):
                for i in range(len(w_padded) - n + 1):
                    char_ngram = w_padded[i:i+n]
                    # Hash 并映射到 [vocab_size, vocab_size + bucket_size - 1] 空间
                    h = fnv1a_hash(char_ngram) % self.bucket_size
                    feature_ids.append(self.vocab_size + h)
                    
        # 2. 获取词级别的 n-gram ID
        if self.word_ngrams > 1:
            for n in range(2, self.word_ngrams + 1):
                for i in range(len(words) - n + 1):
                    # 组合相邻的词，例如 "this movie"
                    word_ngram = " ".join(words[i:i+n])
                    h = fnv1a_hash(word_ngram) % self.bucket_size
                    feature_ids.append(self.vocab_size + h)
                    
        return feature_ids

In [3]:
# In[3]: 定义模型架构

class AuthenticFastTextPyTorch(nn.Module):
    def __init__(self, vocab_size, bucket_size, embed_dim, num_classes):
        super(AuthenticFastTextPyTorch, self).__init__()
        # Embedding 矩阵的总大小 = 词汇表大小 + 哈希桶大小
        total_embeddings = vocab_size + bucket_size
        
        self.embedding = nn.EmbeddingBag(num_embeddings=total_embeddings, 
                                         embedding_dim=embed_dim, 
                                         mode='mean')
        
        # 分类头
        self.fc = nn.Linear(embed_dim, num_classes)
        
    def forward(self, text_ids, offsets):
        embedded = self.embedding(text_ids, offsets) 
        out = self.fc(embedded) 
        return out

In [ ]:
# In[4]: 训练模型

# 训练数据
train_data = [
    ("This movie is fantastic and incredibly good", 1),
    ("I love this wonderful tutorial", 1),
    ("Terrible and incredibly bad garbage", 0),
    ("I hate this awful experience", 0)
]

# 构建精准词表 (真实场景下会统计词频并截断低频词，在词嵌入中实现过截断逻辑)
vocab = {}
idx = 0
for text, _ in train_data:
    for word in text.lower().split():
        if word not in vocab:
            vocab[word] = idx
            idx += 1

# 超参数
VOCAB_SIZE = len(vocab)
BUCKET_SIZE = 100000  # 演示用 10万，真实工程通常用 200万
EMBED_DIM = 100
NUM_CLASSES = 2
EPOCHS = 30

# 初始化特征提取器和模型
extractor = FastTextFeatureExtractor(vocab, bucket_size=BUCKET_SIZE, minn=3, maxn=6, word_ngrams=2)
model = AuthenticFastTextPyTorch(VOCAB_SIZE, BUCKET_SIZE, EMBED_DIM, NUM_CLASSES)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

# 训练循环
for epoch in range(EPOCHS):
    total_loss = 0
    all_ids = []
    offsets = []
    labels = []
    
    current_offset = 0
    for text, label in train_data:
        # 获取融合了 Word, Char N-gram, Word N-gram 的特征 ID
        ids = extractor.get_features(text)
        all_ids.extend(ids)
        offsets.append(current_offset)
        labels.append(label)
        current_offset += len(ids)
        
    all_ids_tensor = torch.tensor(all_ids, dtype=torch.long)
    offsets_tensor = torch.tensor(offsets, dtype=torch.long)
    labels_tensor = torch.tensor(labels, dtype=torch.long)
    
    optimizer.zero_grad()
    predictions = model(all_ids_tensor, offsets_tensor)
    loss = criterion(predictions, labels_tensor)
    loss.backward()
    optimizer.step()
    
    total_loss += loss.item()
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss:.4f}")

print("PyTorch FastText 训练完成！")

Epoch 5/10, Loss: 0.6616
Epoch 10/10, Loss: 0.6198
PyTorch FastText 训练完成！


# 使用官方 fasttext 库

这里调用库简单实现：

In [8]:
%pip install "numpy<2" fasttext

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 114.0 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_versio

In [5]:
# In[5]: 准备 FastText 官方格式的训练文件
# 官方库要求输入文件必须是一个文本文件，每行一条数据，标签前必须带上 '__label__' 前缀。

import fasttext

train_file_path = "fasttext_train.txt"

# 将刚才的数据写入文件
with open(train_file_path, "w", encoding="utf-8") as f:
    for text, label in train_data:
        # 格式示例: __label__1 This movie is fantastic...
        f.write(f"__label__{label} {text}\n")
        
print(f"训练文件已生成: {train_file_path}")

训练文件已生成: fasttext_train.txt


In [6]:
# In[6]: 使用官方库训练和预测

# 训练模型
# wordNgrams=2 告诉模型考虑词级别的 2-gram 组合
# minn 和 maxn 控制字符级别 n-gram 的长度 (默认 minn=3, maxn=6)
print("开始训练官方 FastText 模型...")
ft_model = fasttext.train_supervised(input=train_file_path, epoch=25, lr=0.1, wordNgrams=2)

# 测试预测
test_texts = [
    "This is an incredibly fantastic movie",  # 应该偏向 1
    "What a bad and terrible movie"           # 应该偏向 0
]

print("\n--- 预测结果 ---")
for text in test_texts:
    # predict() 返回 (标签元组, 概率元组)
    labels, probabilities = ft_model.predict(text)
    print(f"文本: '{text}'")
    print(f"预测标签: {labels[0]}, 概率: {probabilities[0]:.4f}\n")

开始训练官方 FastText 模型...

--- 预测结果 ---
文本: 'This is an incredibly fantastic movie'
预测标签: __label__1, 概率: 0.5000

文本: 'What a bad and terrible movie'
预测标签: __label__0, 概率: 0.5000

